In [ ]:
from collections import defaultdict
from glob import glob

import geopandas
import matplotlib.pyplot as plt
import numpy
import pandas

from snail.intersection import GridDefinition
from snail.damages import PiecewiseLinearDamageCurve
from tqdm.notebook import tqdm

In [ ]:
def nonzero_exposure(exposure, hazard_prefix):
    cols = [col for col in exposure.columns if hazard_prefix in col]
    for col in cols:
        exposure.loc[exposure[col] < 0, col] = 0
    exposure[f"{hazard_prefix}_max"] = exposure[cols].max(axis=1)
    return exposure


def cell_index(df, grid):
    col_idx, row_idx = df.i_0.values, df.j_0.values
    nrows, ncols = grid.height, grid.width
    idx = numpy.ravel_multi_index((row_idx, col_idx), (nrows, ncols))
    return idx

# Read buildings with RP exposure
rp_exposure = geopandas.read_parquet("../outputs/exposure/buildings__rp_depths.parquet")

# Clean negative depths to 0, calculate max across RPs
rp_exposure = nonzero_exposure(rp_exposure, "flrf")
rp_exposure = nonzero_exposure(rp_exposure, "flsw")
# Filter to only those with some exposure
rp_exposure = rp_exposure[((rp_exposure.flrf_max > 0) | (rp_exposure.flsw_max > 0))]

# Add integer cell index
grid = GridDefinition.from_raster(
    "../inputs/fluvial_raw_fld_depth/JM_FLRF_UD_Q20_RD_02-aligned.tif"
)
rp_exposure["cell_index"] = cell_index(rp_exposure, grid)

# Clean parish name (to "St. James", "Westmoreland")
rp_exposure.PARISH = (
    rp_exposure.PARISH.str.replace(" ", "").str.replace("ST.", "ST. ").str.title()
)

# Set cell index as index for later joining depths
rp_exposure.set_index("cell_index", inplace=True)

In [ ]:
# Keep only building-relevant columns for later aggregation, cost calculation
exposed_buildings = rp_exposure[
    [
        "osm_id",
        "ED_ID",
        "ED",
        "PARISH",
        "CONST_NAME",
        "building_type",
        "min_damage_cost",
        "max_damage_cost",
        "mean_damage_cost",
        "cost_unit",
        "total_GDP",
        "T500_INTID",
        "geometry",
    ]
].copy()

# Calculate area
exposed_buildings["exposed_area_m2"] = exposed_buildings.area

# Ensure mean is mean of min/max
exposed_buildings.mean_damage_cost = (exposed_buildings.min_damage_cost + exposed_buildings.max_damage_cost) / 2

# Calculate unit repair/rehabilitation cost in Jamaican dollars (from JD/m2 to JD)
cost_cols = ["min_damage_cost", "mean_damage_cost", "max_damage_cost"]
exposed_buildings.loc[:, cost_cols] = exposed_buildings.loc[:, cost_cols] * exposed_buildings.exposed_area_m2.values[:, None]

# Update unit
exposed_buildings["cost_unit"] = "JD"

# Read simplified sector for damage curve mapping
building_sector = pandas.read_csv(
    "../inputs/damage_curves/asset_damage_curve_mapping.csv"
)[["asset_name", "asset_sheet"]]
curve_sector_lookup = defaultdict(list)
for b in building_sector.itertuples():
    curve_sector_lookup[b.asset_sheet].append(b.asset_name)

# Apply simplified sector
for sector in ("industrial", "commercial", "residential"):
    row_mask = exposed_buildings.building_type.isin(curve_sector_lookup[sector])
    exposed_buildings.loc[row_mask, "building_sector"] = sector

In [ ]:
# Join HAZ
haz = (
    geopandas.read_file("../inputs/event_points_hydrological_units/JM_HAZ_T500_02.shp")
    [["T500_ID", "T500_INTID"]]
)
exposed_buildings = exposed_buildings.reset_index().merge(haz, how='left', on="T500_INTID").drop(columns="T500_INTID").set_index("cell_index")

In [ ]:
exposed_buildings.index.name, list(exposed_buildings.columns)

In [ ]:
len(exposed_buildings)

### Assign depths to exposed assets

In [ ]:
def read_event_depths(dataset_path, haz, event_ids):
    for event_id in event_ids:
        try:
            # take max over river/surface flood hazards for a given event
            event_depths = (
                pandas.read_parquet(f"{dataset_path}/T500_ID={haz}/event={event_id}")
                [["cell_index", "depth"]]
                .groupby("cell_index").max()
                .rename(columns={"depth": event_id})
            )
        except FileNotFoundError as e:
            print(e)
            yield None
            continue
        yield event_depths


def join_event_depths(exposed_assets, event_depths):
    reindexed = [exposed_assets]
    for event_depth in event_depths:
        if event_depth is not None:
            reindexed.append(event_depth.reindex(exposed_assets.index))

    assets_with_depths = pandas.concat(reindexed, axis=1)
    return assets_with_depths

In [ ]:
def read_events_metadata(event_set_name):
    river_events = pandas.read_csv(f"../outputs/{event_set_name}_river.csv")
    river_events['hazard'] = 'FLRF'
    precip_events = pandas.read_csv(f"../outputs/{event_set_name}_precip.csv")
    precip_events['hazard'] = 'FLSW'
    events = pandas.concat([river_events, precip_events]).set_index('T500_ID')
    return events

In [ ]:
def assign_haz_asset_depths(event_set_name, events, assets):
    dataset_path = f"../outputs/{event_set_name}"

    haz_ids = events.index.unique()
    assets_by_haz = assets.reset_index().set_index("T500_ID")

    for haz_id in tqdm(haz_ids):
        event_ids = sorted(set(events.loc[haz_id, 'event.id'].unique()))
        haz_assets = assets_by_haz.loc[haz_id].drop(columns="geometry").copy().reset_index().set_index("cell_index")
        event_depths = read_event_depths(dataset_path, haz_id, event_ids)
        asset_event_depths = join_event_depths(haz_assets, event_depths)
        yield asset_event_depths


In [ ]:
obs_name = "ObsEventRP"
obs_events = read_events_metadata(obs_name)
output_dataset_path = f"../outputs/{obs_name}_depths"
for obs_event_depths in assign_haz_asset_depths(obs_name, obs_events, exposed_buildings):
    obs_event_depths.to_parquet(output_dataset_path, partition_cols=["T500_ID"])

In [ ]:
sim_name = "SimEventRP"
sim_events = read_events_metadata(sim_name)
output_dataset_path = f"../outputs/{sim_name}_depths"
for sim_event_depths in assign_haz_asset_depths(sim_name, sim_events, exposed_buildings):
    sim_event_depths.to_parquet(output_dataset_path, partition_cols=["T500_ID"])

### Read damage curves

In [ ]:
curves = {}
for sector in ("industrial", "commercial", "residential"):
    for sensitivity in ("damage_ratio", "damage_ratio_min", "damage_ratio_max"):
        curves[(sector, sensitivity)] = PiecewiseLinearDamageCurve.from_excel(
            "../inputs/damage_curves/damage_curves_buildings_flooding.xlsx",
            sector,
            intensity_col="flood_depth",
            damage_col=sensitivity,
        )

### Calculate event damage

In [ ]:
dfs = []
for fname in sorted(glob(f"../outputs/{obs_name}_depths/T500_ID=*/*")):
    df = pandas.read_parquet(fname)
    dfs.append(df)
obs_event_damage = pandas.concat(dfs)
obs_event_damage.head(2)

In [ ]:
event_cols = [col for col in obs_event_damage.columns if "R06" in col]
for sector in ("industrial", "commercial", "residential"):
    row_mask = obs_event_damage.building_type.isin(curve_sector_lookup[sector])
    depths = obs_event_damage.loc[row_mask, event_cols]
    damage_ratios = curves[(sector, "damage_ratio")].damage_fraction(depths)
    damage_costs = (
        damage_ratios * obs_event_damage.loc[row_mask, "mean_damage_cost"].values[:, None]
    )
    obs_event_damage.loc[row_mask, event_cols] = damage_costs

In [ ]:
building_event_damage = (
    obs_event_damage.drop(
        columns=[
            "min_damage_cost",
            "max_damage_cost",
            "mean_damage_cost",
            "cost_unit",
            "total_GDP",
            "exposed_area_m2",
            # "T500_ID",
        ]
    )
    .groupby(["osm_id", "ED_ID", "ED", "PARISH", "CONST_NAME", "building_type", "building_sector"])
    .sum()
    .reset_index()
)

### Aggregation and plotting

In [ ]:
parish_event_damage = (
    building_event_damage.drop(
        columns=[
            "osm_id",
            "ED_ID",
            "ED",
            "CONST_NAME",
            "building_type",
            "building_sector",
        ]
    )
    .groupby(["PARISH"])
    .sum()
)

In [ ]:
parishes = geopandas.read_file("../inputs/admin_boundaries.gpkg", layer="admin1")[
    ["PARISH", "geometry"]
].set_index("PARISH")

In [ ]:
parish_event_damage_geo = parishes.join(parish_event_damage)

In [ ]:
!mkdir -p ../outputs/figures

In [ ]:
track_ids = []
obs_events = pandas.read_csv("../inputs/event_data/ObsEventInfo.csv")
for track_str in obs_events["track.id"].dropna().values:
    tracks = track_str.split(", ")
    track_ids.extend(tracks)
obs_event_track_ids = obs_events[
    ["event.id", "track.id", "start.year", "start.month", "duration"]
].set_index("event.id")
obs_event_track_ids

In [ ]:
len(obs_events['event.id'].unique()), len(event_cols)

In [ ]:
track_info = pandas.read_csv(
    "../inputs/event_data/tracks_na.tsv",
    sep="\t",
    na_values=None,
    keep_default_na=False,
)
track_info = track_info[track_info["track.id"].isin(track_ids)].set_index("track.id")
track_info.to_csv("../inputs/event_data/track_info.csv")

In [ ]:
for event in event_cols:
    event_id = event.replace("ObsEventRP__FLRF__", "")
    track_str = obs_event_track_ids.loc[event_id, "track.id"]
    try:
        len(track_str)
        track_ids = track_str.split(", ")
        track_names = [track_info.loc[track_id, "name"] for track_id in track_ids]
        title = f"{event_id}\n{', '.join(track_ids)} ({', '.join(track_names)})"
    except:
        title = event_id

    fig, ax = plt.subplots()
    ax.set_title(title)
    parish_event_damage_geo.plot(
        ax=ax,
        column=event,
        legend=True,
        legend_kwds={
            "label": "River flooding direct damage to buildings (J$)",
            "orientation": "horizontal",
        },
    )
    ax.set_axis_off()
    plt.savefig(f"../outputs/figures/{event}_parish_building_damages.png")
    plt.close()

In [ ]:
total_event_damage = building_event_damage.drop(
    columns=[
        "osm_id",
        "ED_ID",
        "ED",
        "CONST_NAME",
        "building_type",
        "building_sector",
        "PARISH",
    ]
).sum()
total_event_damage = pandas.DataFrame(total_event_damage).reset_index()
total_event_damage.columns = ["event.id", "total_building_damage_JD"]
total_event_damage["event.id"] = total_event_damage["event.id"].str.replace(
    "ObsEventRP__FLRF__", ""
)
total_event_damage = total_event_damage.sort_values(by='event.id').set_index('event.id')
total_event_damage = total_event_damage.join(obs_event_track_ids).reset_index()

def title(e):
    try:
        track_ids = e['track.id'].split(", ")
        track_names = []

        for track_id in track_ids:
            try:
                track_names.append(track_info.loc[track_id, "name"])
            except KeyError:
                pass
    except AttributeError:
        track_names = []
    return f"{', '.join(track_names)} {e['start.year']}-{e['start.month']:02}"
total_event_damage["event.title"] = total_event_damage.apply(title, axis=1)
total_event_damage

In [ ]:
total_event_damage.set_index("event.title")[["total_building_damage_JD"]].plot.bar()